# AzSemBench — Existing Model Paraphrase Evaluation


Evaluates pretrained sentence encoders without updating weights. Threshold is selected on dev only, then fixed for train/test/overall.

Run top to bottom in Colab, Kaggle (Internet + GPU enabled), or local Jupyter. Outputs are written under `outputs/`. See the repository README for metrics, supplied results and dataset filtering.


In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "scipy>=1.12,<2" "scikit-learn>=1.4,<2" "sentence-transformers>=3.4,<6" "transformers>=4.48,<5" "datasets>=3,<5" "accelerate>=1,<2" "torch>=2.4,<3" "sentencepiece>=0.2,<1" "protobuf>=4,<7" "rapidfuzz>=3,<4" "python-dotenv>=1,<2"


## 1. Download model-ready data from GitHub


In [ ]:
# Configuration: environment variables override these defaults.
import os
from pathlib import Path
from urllib.request import urlretrieve
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
REPO = os.getenv("AZSEMBENCH_REPO", "MehribanAliyeva/azsembench")
REF = os.getenv("AZSEMBENCH_REF", "main")  # use a commit SHA for a fixed dataset
DATA_URL = f"https://raw.githubusercontent.com/{REPO}/{REF}/data/model_ready/core_model_ready.csv"
DATA_DIR = Path("data/model_ready")
DATA_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = DATA_DIR / "core_model_ready.csv"
# Download on each run so data is explicitly tied to REPO/REF.
try:
    urlretrieve(DATA_URL, CSV_PATH)
except Exception as exc:
    raise RuntimeError(f"Cannot download {DATA_URL}. Publish the prepared repo first and enable Internet.") from exc
import hashlib
print("Dataset:", DATA_URL)
print("SHA256:", hashlib.sha256(CSV_PATH.read_bytes()).hexdigest())


## 2. Imports and configuration


In [ ]:
import os
import gc
from pathlib import Path
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,roc_auc_score
MODELS=['sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2','sentence-transformers/paraphrase-multilingual-mpnet-base-v2','sentence-transformers/LaBSE','LocalDoc/az-en-MiniLM-L6-v2','intfloat/multilingual-e5-large']
OUTPUT_DIR=Path(os.getenv("AZSEMBENCH_OUTPUT_DIR", "outputs/paraphrase_detection")); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
BATCH_SIZE=32
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:',DEVICE)


## 3. Helpers


In [ ]:
def clean(x): return '' if x is None or pd.isna(x) else str(x).strip()
def filter_para(df):
    mask=pd.Series(False,index=df.index)
    if 'task' in df.columns: mask |= df['task'].astype(str).str.lower().eq('paraphrase_detection')
    if 'recomputed_task' in df.columns: mask |= df['recomputed_task'].astype(str).str.lower().eq('paraphrase_detection')
    if 'source_dataset' in df.columns: mask |= df['source_dataset'].astype(str).str.lower().str.contains('paws',regex=False)
    out=df.loc[mask].copy()
    if out.empty: raise RuntimeError('No paraphrase-detection rows found.')
    return out
def prep(name,texts): return [f'query: {t}' for t in texts] if 'e5' in name.lower() else texts
def choose_threshold(y, s):
    """Maximize dev F1, break ties by accuracy; sorted cumulative counts."""
    y = np.asarray(y, int); s = np.asarray(s, float)
    if len(y) == 0 or not np.isfinite(s).all() or not np.isin(y, [0, 1]).all():
        raise ValueError("Threshold selection requires nonempty finite scores and binary labels.")
    order = np.argsort(s, kind="stable"); sy = y[order]; ss = s[order]
    unique, starts = np.unique(ss, return_index=True)
    if len(unique) == 1:
        candidates = unique
        cuts = np.array([0])
    else:
        candidates = np.r_[unique[0]-1e-6, (unique[:-1]+unique[1:])/2, unique[-1]+1e-6]
        cuts = np.r_[0, starts[1:], len(y)]
    positives = np.r_[0, np.cumsum(sy)]
    tp = positives[-1] - positives[cuts]
    fp = len(y) - cuts - tp
    fn = positives[cuts]
    tn = cuts - fn
    denominator = 2*tp + fp + fn
    f1s = np.divide(2*tp, denominator, out=np.zeros(len(cuts), float), where=denominator != 0)
    accuracies = (tp+tn)/len(y)
    best_t, best_f, best_a = None, -1., -1.
    for t, f, a in zip(candidates, f1s, accuracies):
        if f > best_f or (np.isclose(f, best_f) and a > best_a):
            best_t, best_f, best_a = float(t), float(f), float(a)
    return best_t, best_f

def metrics(y,s,t):
    p=(s>=t).astype(int)
    return {'accuracy':accuracy_score(y,p),'f1':f1_score(y,p,zero_division=0),'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),'roc_auc':roc_auc_score(y,s) if len(np.unique(y))==2 else np.nan}


## 4. Load data


In [ ]:
df=pd.read_csv(CSV_PATH,dtype=object,keep_default_na=False,encoding='utf-8-sig')
for c in ['sentence1_az','sentence2_az','label','split']:
    if c not in df.columns: raise RuntimeError(f'Missing required column: {c}')
data=filter_para(df)
data['sentence1_az']=data['sentence1_az'].map(clean); data['sentence2_az']=data['sentence2_az'].map(clean)
data=data[(data['sentence1_az']!='')&(data['sentence2_az']!='')].copy()
data['label']=pd.to_numeric(data['label'],errors='raise')
if not data['label'].isin([0, 1]).all(): raise ValueError('Paraphrase labels must be binary.')
data['label']=data['label'].astype(int)
data['split']=data['split'].astype(str).str.lower().str.strip()
if 'dev' not in set(data['split']): raise RuntimeError('No dev split found.')
print('Rows:',len(data)); print(data['split'].value_counts()); print(data['label'].value_counts().sort_index())

data["split"] = data["split"].astype(str).str.lower().str.strip()
if set(data["split"]) != {"train", "dev", "test"}:
    raise ValueError("Expected nonempty train/dev/test splits.")
if not data["label"].isin([0, 1]).all(): raise ValueError("Paraphrase labels must be binary.")


## 5. Evaluate models


In [ ]:
s1=data['sentence1_az'].tolist(); s2=data['sentence2_az'].tolist()
result_rows=[]; threshold_rows=[]; pred_frames=[]
for name in MODELS:
    print('\nEvaluating',name)
    model=None
    try:
        model=SentenceTransformer(name,device=DEVICE)
        e1=model.encode(prep(name,s1),batch_size=BATCH_SIZE,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
        e2=model.encode(prep(name,s2),batch_size=BATCH_SIZE,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True)
        sim=np.sum(e1*e2,axis=1)
        work=data.copy(); work['_sim']=sim
        d=work[work['split'].eq('dev')]
        t,dev_f1=choose_threshold(d['label'].to_numpy(int),d['_sim'].to_numpy(float))
        threshold_rows.append({'model':name,'threshold':t,'selection_split':'dev','selection_metric':'f1','dev_f1':dev_f1})
        for split in list(work['split'].unique())+['overall']:
            sub=work if split=='overall' else work[work['split'].eq(split)]
            result_rows.append({'model':name,'split':split,'n':len(sub),'threshold':t,**metrics(sub['label'].to_numpy(int),sub['_sim'].to_numpy(float),t)})
        pred=(sim>=t).astype(int)
        pred_frames.append(pd.DataFrame({'model':name,'split':data['split'].values,'source_id':data['source_id'].astype(str).values if 'source_id' in data.columns else data.index.astype(str),'gold_label':data['label'].values,'predicted_label':pred,'cosine_similarity':sim,'threshold':t,'sentence1_az':data['sentence1_az'].values,'sentence2_az':data['sentence2_az'].values}))
        display(pd.DataFrame([r for r in result_rows if r['model']==name]))
    finally:
        if model is not None: del model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


## 6. Save and download


In [ ]:
results=pd.DataFrame(result_rows); thresholds=pd.DataFrame(threshold_rows); preds=pd.concat(pred_frames,ignore_index=True)
results.to_csv(OUTPUT_DIR/'paraphrase_detection_results.csv',index=False,encoding='utf-8-sig')
thresholds.to_csv(OUTPUT_DIR/'paraphrase_detection_thresholds.csv',index=False,encoding='utf-8-sig')
preds.to_csv(OUTPUT_DIR/'paraphrase_detection_predictions.csv',index=False,encoding='utf-8-sig')
display(results[results['split'].eq('test')].sort_values('f1',ascending=False))
from IPython.display import FileLink
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    display(FileLink(str(path)))
